# Sesión 4 · Transformación de datos — empleos (versión SIN cola larga (outliers de duración eliminados))

**Curso:** Minería de Datos · UNIMINUTO Ibagué · 2026-2
**Ingreso:** salida de `3.0_preprocesamiento.ipynb` · **Paso (CRISP-DM):** Preparación de datos → *transformación*.

> Regla de oro de la sesión: **no se puede entrenar una IA con datos no numéricos**. Los valores categóricos, si se entregan tal cual, la sesgan. Aquí llevamos cada columna a una representación numérica *honesta* (one-hot u ordinal) y luego **escalamos** las numéricas para que ninguna columna "secuestre" las distancias que usarán KNN / K-means en sesiones posteriores.
>
> Desde la integración de ESCO el dataset trae además **competencias por oficio**: la sección **2.5** las pasa a número, deja por qué de las seis entran solo dos a la `X`, y documenta el costo de hacerlo (aquí se pierden 26.186 filas, el 7,68 % sin oficio clasificado).
>
> Este cuaderno es **transversal a las dos versiones** de la sesión 3. En la celda de configuración de la sección 0 se elige `con_outliers` (376.567 filas) o `sin_outliers` (341.090 filas). La sección 4 re-audita los outliers en tiempo de ejecución para decidir el escalador.

## 0 · Entorno, versión y carga

```text
Limpio -> Codificar categóricas -> Escalar -> (PCA si aplica) -> Modelar
```
Hoy cubrimos los tres primeros bloques.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

# Regla D5: la raiz se resuelve buscando 'data/' hacia arriba; nunca una ruta
# absoluta de esta maquina (mismo criterio que src/limpieza/*.py y que 2.0 / 3.0).
RAIZ = Path.cwd()
while not (RAIZ / "data").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

# El calculo vive en src/transformacion/, no en este cuaderno: aca esta la
# narrativa y la evidencia; alla, el codigo. Este cuaderno pasa a ser la
# evidencia de lo que el modulo produce, no una copia del modulo.
sys.path.insert(0, str(RAIZ / "src" / "transformacion"))
import diagnostico as D
import transformacion as T

print("Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| sklearn", sklearn.__version__)

# <<< CONFIG: cambie el valor de VERSION para elegir la versión del dataset >>>
VERSION = "sin_outliers"          # "con_outliers"  o  "sin_outliers"
CSV = T.ruta_csv(VERSION)
print("\nRaíz del proyecto:", RAIZ)
print("Versión:", VERSION, "| archivo:", CSV.name)

# Una sola semilla y un solo objeto rng, que se consume en orden: primero la
# submuestra de trabajo y despues el PCA. Si cada paso creara el suyo, el PCA
# recibiria un estado distinto y la corrida no seria reproducible (regla D5).
rng = np.random.RandomState(T.SEED_MUESTRA)

Python 3.14.7 | pandas 3.0.2 | numpy 2.4.4 | sklearn 1.9.1

Raíz del proyecto: C:\Users\ACER\Desktop\MineriaProtect
Versión: sin_outliers | archivo: empleos_limpio_sin_outliers.csv


In [2]:
df = T.leer(VERSION)
print(f"dataset           : {df.shape[0]:,} filas x {df.shape[1]} columnas | "
      f"{df['resume_id'].nunique():,} personas")
print("\nNulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):")
print(df.isna().sum()[df.isna().sum() > 0].to_string())

dataset           : 341,090 filas x 20 columnas | 69,182 personas

Nulos por columna (los NaN de ocupacion ya estan explicados por banderas en la sesion 3):
occupation_code        26186
occupation_label       26186
isco_group             23886
isco_group_label       23886
isco_level             23886
n_skills_essential     26186
n_skills_competence    26186
n_skills_knowledge     26186
n_skills_sin_tipo      26186
veces_ese_oficio       26186


---
## 1 · ¿Por qué cada columna es *one-hot* u *ordinal*?

La naturaleza de la variable decide la codificación. La regla del curso:

- **Nominales** (sin jerarquía natural) → **one-hot**: "aseo, libros, licores" no tienen orden, no se puede decir que uno "vale más" que otro. Numerarlos 1, 2, 3 inventaría distancia.
- **Ordinales** (jerarquía real y verificable) → **ordinal / KBins**: la duración de un empleo o el nivel educativo sí tienen orden ("jefe > auxiliar").

Las cifras de filas dependen de la versión elegida; todo se calcula en tiempo de ejecución.

| Columna | Naturaleza | Codificación elegida | Por qué |
|---|---|---|---|
| `start_date` / `end_date` | **Ordinal** (tiempo: Q1 1965 < Q1 2026) | ordinal (año×4+q) → `KBinsDiscretizer` (demo) + escalado | El tiempo tiene orden estricto y significado; jamás one-hot (≈206 categorías y se perdería el orden) |
| `university_level` | **Ordinal** (jerarquía de estudios) | mapeo ordinal manual 0–4 | "No reportado < Secondary school < Bachelor < Master < PhD" es jerarquía real; son solo 5 valores discretos, mapeo directo (KBins no aplica a 5 niveles fijos) |
| `isco_level` | **Ordinal por diseño** (nivel 1–4 ISCO) | → **se descarta** | En los datos es prácticamente **constante** (`'4'` en ~93% y `NaN` donde la ocupación es unknown). Una columna sin variabilidad no discrimina. Dentro de la `X` es **constantemente `'4'`, con 0 nulos**, así que el descarte es aún más fácil. La bandera `es_unknown_ocupacion` que «capturaba» su ausencia en el dataset completo es justamente una de las que la 2.5 deja constante y por eso excluye de la `X`: ver 2.5 |
| `emparejado` | **Nominal** (ok / unknown / rescatado) | **one-hot** (`get_dummies`) | Son 3 estados sin orden: "rescatado" no es mayor que "ok". **Ojo: tras el drop de la 2.5 quedan solo filas `ok`, así que `get_dummies` devuelve 1 columna (`emparejado_ok`), no 3** |
| `matched_code` | **Nominal** (código con el que se emparejó) | **one-hot en esencia → se fusiona** | Coincide 100% con `occupation_code` cuando `emparejado='ok'` y guarda el código recuperado en `rescatado`; se fusiona en `occupation_code` (ver celda abajo) |
| `occupation_code` | **Nominal** (≈2.856 códigos) | **one-hot** (`get_dummies`) | Códigos de ocupación ISCO son etiquetas sin orden. Ojo: son 2.855 categorías → *explosión de columnas* (se analiza abajo) |
| `occupation_label` | **Nominal** (nombre del cargo) | → **se descarta** | Es 1:1 con `occupation_code` (una etiqueta por código): duplicaría las 2.856 columnas sin información nueva |
| `isco_group` / `isco_group_label` | **Nominal** (grupo ISCO 4 dígitos) | → **se descartan** | `isco_group` es un *agregado* del código de ocupación (sus 4 primeros dígitos) y `isco_group_label` es 1:1 con él; uno-hotear `occupation_code` ya los contiene |
| `es_unknown_ocupacion`, `es_rescatado`, `es_vigente` | **Binarias** (flags True/False) | **solo `es_vigente` entra** (0/1 directo, equivale a one-hot con `drop='first'`); las otras dos **se excluyen de la `X`** (ver 2.5) | Ya son binarias: escalarlas no aporta y rompe su legibilidad (slide 21). Las dos que se excluyen quedan constantes `False` en este universo: una columna constante no aporta a la distancia y falsea la importancia de variables |
| `resume_id` | Identificador | **se descarta** | Es la clave primaria (una por persona), no un patrón |

### 1.1 Verificación de los descartes (auditoría de redundancia)

Los descartes NO son opinión: se demuestran con los datos.


In [3]:
# Las banderas y el cruce con ESCO tienen que cuadrar ANTES de construir la X:
# si el cruce no fuera 1:1, la tabla de competencias quedaria mal atribuida.
cruce = D.comprobaciones_cruce(df)
print("matched_code == occupation_code cuando emparejado='ok':", cruce["code_coincide"])
print("occupation_label x code (max etiquetas por codigo, 1 = 1:1):", cruce["max_etiquetas_por_codigo"])
print("isco_group_label x group (max etiquetas por grupo, 1 = 1:1):", cruce["max_etiquetas_por_grupo"])
print("isco_level valores distintos:", cruce["isco_level_valores"])
print("NaN de isco_level == es_unknown_ocupacion=True:", cruce["isco_nan_coincide"])

print("\nmatched_code en emparejado='unknown':", cruce["matched_unknown"])
print("matched_code en emparejado='rescatado' (codigos recuperados):", cruce["matched_rescatado"])

matched_code == occupation_code cuando emparejado='ok': 1.0
occupation_label x code (max etiquetas por codigo, 1 = 1:1): 1
isco_group_label x group (max etiquetas por grupo, 1 = 1:1): 1
isco_level valores distintos: ['4']
NaN de isco_level == es_unknown_ocupacion=True: 1

matched_code en emparejado='unknown': <ArrowStringArray>
['unknown']
Length: 1, dtype: str
matched_code en emparejado='rescatado' (codigos recuperados): <ArrowStringArray>
['2659.2.2', '3412.4.9', '2424.3']
Length: 3, dtype: str


### 1.2 Fusión: `occupation_code` conserva el código recuperado de `rescatado`

Para `rescatado` el código vive en `matched_code`; para `unknown` ambas están vacías/`'unknown'`. Fusionamos para dejar **una sola** columna de ocupación.


In [4]:
# Las filas rescatadas en la sesion 3 llegaron sin `occupation_code` (solo con
# `matched_code`). Se les repone el codigo recuperado y dejan de ser "sin ocupacion".
df = T.fusionar_rescatado(df)
print("NaN restantes de ocupacion tras fusion:", int(df["occupation_code"].isna().sum()),
      "| deben coincidir con emparejado='unknown':",
      int((df["occupation_code"].isna() == (df["emparejado"] == "unknown")).all()))

NaN restantes de ocupacion tras fusion: 23886 | deben coincidir con emparejado='unknown': 1


---
## 2 · Codificación ordinal

### 2.1 Fechas → ordinal numérico

`Q1 2011` es texto. Lo llevamos a `año × 4 + trimestre` (Q1 2011 = 2011×4+1 = 8.045), que preserva el orden temporal y permite operaciones aritméticas.


In [5]:
# 'Q3 2015' no es comparable con 'Q4 2015' como texto. Se pasa a un ordinal
# (anio * 4 + trimestre) para que la aritmetica valga. 'Present' es censura
# por la derecha: no se inventa su fin, se ubica en Q1 2026.
df = T.fechas_ordinales(df)
print("start_ord : min", df["start_ord"].min(), "max", df["start_ord"].max())
print("end_ord   : min", df["end_ord"].min(), "max", df["end_ord"].max(),
      "| 'Present' reubicados:", int((df["end_date"] == "Present").sum()))
print("dur_q     : min", df["dur_q"].min(), "max", df["dur_q"].max())

start_ord : min 7869 max 8085
end_ord   : min 7881.0 max 8105.0 | 'Present' reubicados: 18956
dur_q     : min 1.0 max 212.0


### 2.2 `university_level` → mapeo ordinal manual

Son **5 niveles discretos con jerarquía real**; un mapeo manual explícito es más fiel que discretizar (KBins agruparía niveles arbitrariamente). Documentamos la escala en el propio código.


In [6]:
# La ausencia es un estado, no un cero engañoso de estudios: 'No reported' es el
# nivel mas bajo de la escala ordinal, no "sin educacion = 0 estudios".
df = T.educacion_ordinal(df)
print("univ_ord unicos:", sorted(df["univ_ord"].unique().tolist()))
print("orden verificado:", [u for u in sorted(df["univ_ord"].unique())])

univ_ord unicos: [0, 1, 2, 3, 4]
orden verificado: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### 2.3 `isco_level`: por qué se descarta (es constante)

Es **ordinal por diseño** (nivel de competencia ISCO 1–4), pero en estos datos no discrimina:
- En el dataset completo, ~93% de las filas vale `4`; el resto es `NaN` = ocupación desconocida.
- Ese `NaN` coincide exactamente con `es_unknown_ocupacion = True`, que es lo que lo «explica».
- **Pero dentro de la `X` el argumento ya no aplica**: la 2.5 se queda solo con filas `saber_skills='ok'`,
  de modo que `isco_level` es `'4'` en el 100 % y con 0 nulos, y `es_unknown_ocupacion` es `False`
  en el 100 %. La columna es constante igual, pero por otra razón: no por las banderas, sino porque
  las filas que la variaban ya no están.

Dejarlo como número casi constante no aporta información al modelo y genera una falsa sensación de variable.


In [7]:
# isco_level tiene un solo valor util -> no aporta nada que isco_group no diga.
tabla_isco, prop_isco = D.revisar_isco(df)
print("Conteo de isco_level:")
print(tabla_isco.to_string())
print(f"\nproporcion de filas == '4': {prop_isco:.4f}  -> columna ~constante, se descarta.")

Conteo de isco_level:
isco_level
4      317204
NaN     23886

proporcion de filas == '4': 0.9300  -> columna ~constante, se descarta.


### 2.4 `KBinsDiscretizer` (método del curso) aplicado a las fechas

El curso lo usa para convertir una continua en **segmentos ordinales** de igual tamaño (*quantile* = ~misma cantidad de observaciones por rango). Lo aplicamos a `start_ord` y `end_ord`.


In [8]:
# Exploracion, NO parte de la X: estos segmentos no entran al final (ver la
# celda del PCA, que explica por que las fechas se quedan continuas).
seg = D.segmentos_cuantiles(df)
print("Bordes de start_ord:", seg["bordes_inicio"].round(0))
print("Bordes de end_ord  :", seg["bordes_fin"].round(0))
print("\nStart en 5 segmentos (mismo tamano por cuantiles):")
print(seg["conteo_inicio"].to_string())

Bordes de start_ord: [7869. 8021. 8041. 8050. 8061. 8085.]
Bordes de end_ord  : [7881. 8029. 8046. 8057. 8067. 8105.]

Start en 5 segmentos (mismo tamano por cuantiles):
0.0    62273
1.0    72194
2.0    68594
3.0    64078
4.0    73951


### 2.5 Las competencias ESCO: `n_skills_essential` y `veces_ese_oficio`

Las 6 columnas de competencias que el pipeline integra llegan como **texto** (el proyecto lee todo
con `dtype=str`, seccion 0). Aqui pasan a numero y se decide cuales entran a la `X`.

**Dos advertencias que condicionan todo lo que sigue** (desarrolladas en `docs/SKILLS.md`):

1. **Atribuyen al CARGO, no a la PERSONA.** ESCO publica que competencias *exige un oficio*, no
   que sabe un individuo: lo que se hace es atribuir al puesto. No hay variabilidad dentro de un
   mismo oficio, asi que esto **no** se puede leer como "esta persona sabe N cosas".
2. **Una parte de las filas no tiene dato.** Cuando el oficio no esta clasificado
   (`saber_skills = 'no_clasificado'`) los contadores vienen **vacios, nunca 0**: un 0 afirmaria
   que el cargo no exige ninguna competencia, que es falso. `sklearn` no acepta `NaN` en un
   escalador y este proyecto no imputa valores, asi que la salida honesta es **dropear esas
   filas** y documentar la perdida. Cuantas son se mide antes de dropear, no despues.

De las 6 columnas entran **dos**. Los otros tres contadores (`_competence`, `_knowledge`,
`_sin_tipo`) son la misma variable repartida y no aportan informacion independiente; y
`n_skills_essential` es funcion determinista de `occupation_code`, o sea que si la `X` ya lleva
`occupation_<codigo>` en one-hot, su informacion ya esta ahi.


In [9]:
# --- 2.5 El universo de trabajo: contadores de ESCO, drop y binario ---
# Regla del proyecto: todo se lee con dtype=str (para no perder los ceros a la
# izquierda de los codigos ESCO). Los contadores son la excepcion y hay que
# convertirlos explicitamente. El costo del drop se mide ANTES, y el modulo lo
# devuelve ya medido en vez de recalcularlo en el cuaderno.
df_antes, df, info = T.universo_con_competencias(df)
print(f"Filas sin dato de competencias: {info['n_sin_dato']:,} "
      f"({100 * info['prop_sin_dato']:.2f} %)")
print("  su composicion (emparejado):")
print(info["composicion"].to_string())
print("\nRegla vazio != 0 -> contadores en 0 (deberia ser 0):", info["ceros"])
print("El vacio coincide exacto con saber_skills == 'no_clasificado':", info["vacio_exacto"])

print(f"\nUniverso de trabajo: {info['filas_antes']:,} -> {info['filas_despues']:,} filas "
      f"({info['filas_despues'] - info['filas_antes']:+,} | "
      f"{100 * (info['filas_despues'] - info['filas_antes']) / info['filas_antes']:+.2f} %)")
print(f"                    {info['personas_antes']:,} -> {info['personas_despues']:,} personas")
print(f"Ocupaciones distintas con dato: {info['oficios']:,}")

# veces_ese_oficio esta muy concentrada en 1: como continua aporta poco mas que
# "volvio o no". Se conservan las DOS formas porque no son lo mismo -> el binario
# es estable frente al escalador, y el conteo distingue 2 repeticiones de 19.
print(f"\nreincide (veces_ese_oficio > 1): {int(df['reincide'].sum()):,} filas "
      f"({100 * df['reincide'].mean():.1f} %) | "
      f"{df.loc[df['reincide'] == 1, 'resume_id'].nunique():,} personas")
print()
for c in T.CONTADORES + T.BINARIA:
    print(f"  {c:<20} min {df[c].min():>6.0f}  max {df[c].max():>6.0f}  "
          f"mediana {df[c].median():>6.1f}  sd {df[c].std():>6.1f}")

Filas sin dato de competencias: 26,186 (7.68 %)
  su composicion (emparejado):
emparejado
unknown      23886
rescatado     2300

Regla vazio != 0 -> contadores en 0 (deberia ser 0): 0
El vacio coincide exacto con saber_skills == 'no_clasificado': True

Universo de trabajo: 341,090 -> 314,904 filas (-26,186 | -7.68 %)
                    69,182 -> 68,338 personas
Ocupaciones distintas con dato: 2,826

reincide (veces_ese_oficio > 1): 99,190 filas (31.5 %) | 31,943 personas

  n_skills_essential   min      4  max     99  mediana   20.0  sd   13.4
  veces_ese_oficio     min      1  max     19  mediana    1.0  sd    1.2
  reincide             min      0  max      1  mediana    0.0  sd    0.5


---
## 3 · Codificación one-hot con `get_dummies`

### 3.1 El problema de las ~2.856 ocupaciones

> *"ciudad × país explota"* — el curso advierte que pocas columnas nominales de alta cardinalidad generan muchísimas columnas. Una ocupación por persona → filas enormemente dispersas.

El grueso de las filas usa ocupaciones únicas. Por eso:

- La demostración con `get_dummies` (matriz densa) se hace sobre una **submuestra representativa de 50.000 filas**; se mantiene la semilla para reproducibilidad.
- Al final de la sección se muestra la alternativa de **producción**: `OneHotEncoder(sparse_output=True)`, que codifica el dataset completo sin explotar la memoria.

Primero, el **bloque de frecuencias**:


In [10]:
# ~2.800 categorias en 348.263 filas: la `X` densa completa no cabe en memoria.
# Se trabaja sobre una submuestra sembrada, y el mismo rng sigue disponible mas
# adelante para el PCA (ver la celda de configuracion).
frec = df["occupation_code"].dropna().value_counts()
print("Categorias unicas (con datos):", int(frec.size), "-> ~2.800 columnas dummy\n")
print(frec.head(10).to_string())

df_s = T.submuestra(df, rng)
print(f"\nSubmuestra de trabajo: {len(df_s):,} filas")

Categorias unicas (con datos): 2826 -> ~2.800 columnas dummy

occupation_code
3343.1      15384
5223.4       9432
5223.6       9005
9333.8       6650
5230.1       5667
9412.1       5440
5131.2       4879
9112.2       4490
9329.1       4231
9333.8.1     3488

Submuestra de trabajo: 50,000 filas


In [11]:
# one-hot nominal. Tras el drop de la 2.5 no queda ninguna fila sin ocupacion,
# asi que tampoco hay ninguna fila en "todo 0" que explicar.
cod = T.codificar(df_s)
print("emparejado ->", cod["emparejado"].shape[1], "columnas", cod["emparejado"].columns.tolist())
print("ocupacion  ->", cod["ocupacion"].shape[1], "columnas")
print("flags      ->", cod["banderas"].shape[1], "columnas", cod["banderas"].columns.tolist())

emparejado -> 1 columnas ['emparejado_ok']
ocupacion  -> 2317 columnas
flags      -> 1 columnas ['es_vigente']


> **Recuerda (slides 18–21 de la sesión 4):** las columnas dummy y binarias **NO se escalan**. Escalarlas las vuelve ilegibles sin ganar nada.

### 3.2 Alternativa de producción: `OneHotEncoder` con salida dispersa (dataset completo)

Mismo resultado, pero usa una matriz *sparse* (guarda solo los 1). Con esto **sí** se puede codificar el dataset completo sin 1–9 GB en RAM.


In [12]:
# Que pasaria sin submuestra: el bloque one-hot sparse del dataset entero.
forma_ohe, bytes_ohe = D.bloque_onehot_completo(df)
print("OneHotEncoder (full, sparse):", forma_ohe)
print("Memoria relativa del bloque cf. densa:", bytes_ohe, "bytes aprx.")

OneHotEncoder (full, sparse): (314904, 2827)
Memoria relativa del bloque cf. densa: 5038464 bytes aprx.


---
## 4 · Escalamiento: ¿cuál de los 3 y por qué?

### 4.1 Los outliers documentados en la sesión 3 (auditoría en tiempo de ejecución)

La sesión 3 trató cada versión de forma distinta: en la versión **con** cola larga conservó los extremos (decisión de negocio); en la versión **sin** cola larga los eliminó (`dur ≥ 25` trimestres). Aquí se re-calculan los límites Tukey (IQR) **sobre la versión elegida**, en las dos variantes de `dur_q` (con `Present` → censura, igual que la sesión 3, y con `Present` → Q1 2026, nuestro encoding):


In [13]:
# dur_q aparece dos veces a proposito: censurado (los vigentes quedan en NaN) y
# ubicado en Q1 2026. La diferencia entre las dos filas es el costo de tratar la
# censura como si tuviera fecha de fin.
#
# Se miden sobre dos universos: las 4 primeras filas sobre `df_antes` (antes del
# recorte por falta de dato de competencias) y la `dur_q` censurada sobre `df`.
# Es como lo midio el cuaderno siempre; ver el docstring de `tabla_tukey`.
D.tabla_tukey(df_antes, df)

,variable,Q1,Q3,IQR,lim_inf,lim_sup,outliers,prop_outliers
0,start_ord,8029.0,8057.0,28.0,7987.0,8099.0,22459,0.066
1,end_ord,8034.0,8065.0,31.0,7987.5,8111.5,17089,0.050
2,dur_q (Present censurado),1.0,8.0,7.0,-9.5,18.5,11306,0.038
3,dur_q (Present = Q1 2026),1.0,8.0,7.0,-9.5,18.5,31042,0.091
4,university_map,1.0,2.0,1.0,-0.5,3.5,1461,0.004


### 4.2 Decisiones de escalador (una por columna)

**Por qué escalar (medido, no supuesto).** KNN y K-means eligen vecinos por distancia euclídea, y la distancia suma diferencias **al cuadrado**: la columna con más unidades secuestra la métrica y las demás quedan mudas. La celda siguiente lo cuantifica sobre nuestra submuestra: cuánto aporta cada columna a la distancia entre `N_PARES` parejas de filas, sin escalar, con cada escalador por separado y con la mezcla que sale a producción. No lo afirmamos de teoría, lo calculamos.

**Lo que revela la Medición.** Escalar no es solo «poner en la misma unidad»: es **decidir a qué variable el modelo le va a hacer caso**. Sin escala, las dos fechas se llevan 83,8 % de la distancia y la educación queda en 0,03 % — un agrupamiento separaría por calendario, no por carrera. Con la mezcla de producción, la duración del empleo (nuestro comportamiento de interés) queda en 41,45 %, y las competencias ESCO —que son justo lo que se agregó en la 2.5— pasan de 8,76 % sin escalar a 14,24 % con su `RobustScaler`.

**Deuda que deja la auditoría (no se corrige en silencio).** `MinMaxScaler` lleva la educación (rango 0–4) a [0, 1], pero `RobustScaler` extiende las fechas por IQR, que es mucho más angosto que su rango total; el resultado es que en la distancia la educación queda por debajo del 1 %. Elegimos cada escalador por el perfil de outliers de su columna, no por su aporte a la distancia: es un criterio válido, pero tiene este efecto lateral. Se documenta como decisión abierta a revisar antes de la fase de minería, junto con D1–D4.

| Columna | Outliers detectados | Escalador | Por qué |
|---|---|---|---|
| `start_ord` | ~22 mil (cola baja de fechas viejas; la limpieza de duración no la tocó) | **RobustScaler** | La mediana y el IQR no se dejan arrastrar por la cola baja |
| `end_ord` | ~17 mil (cola baja + censura de `Present`) | **RobustScaler** | Idem |
| `dur_q` | ~31 mil en nuestro encoding (la censura `Present`→2026 rehace la cola) y ~12 mil con la definición de la sesión 3 | **RobustScaler** | Lección clave: eliminar la cola larga del CSV **no elimina la necesidad** del escalador robusto; la censura y la nueva distribución vuelven a crear extremos |
| `university_map` | rango fijo [0, 4] por diseño, sin atípicos | **MinMaxScaler** | Rango acotado y conocido → el caso natural del MinMax; lleva todo a [0, 1] |
| `n_skills_essential` | sin atípicos (rango 4–99) | **RobustScaler** | Decisión por **escala**, no por outliers: con 94 unidades de rango, sin escalarla duplica el peso de las dos fechas y deja muda a `univ_ord` (rango 4). Se mide en la celda siguiente |
| `veces_ese_oficio` | sin atípicos (rango 1–19) | **RobustScaler** | Idem. Además está muy concentrada (64 % de valores = 1), de ahí que se le acompaña su binario |
| `reincide` (`veces > 1`) | — | *no se escala* | Es 0/1: ya está en su rango. Escalarla la haría ilegible sin ganar nada (misma regla que las dummies, slide 21) |
| `StandardScaler` | — | *no se elige* | Usa media/desviación: sensible a las colas restantes. Se muestra la comparación abajo |

> **Por qué 11.306 y 31.042 difieren (2,7 veces) y cuál es el número bueno.** Las dos filas son el
> mismo criterio de Tukey sobre la **misma** `dur_q`; solo cambia qué se hace con las experiencias
> vigentes, que aquí ya no tienen nada que ver con la cola larga. Censurándolas (`NaN`) quedan
> 11.306 outliers sobre las filas con fin observado. Rellenándolas con Q1 2026 se les inventan
> 21 trimestres a cada vigente y el total se dispara a 31.042. **El número defendible es 11.306**,
> porque no imputa. Nótese que quitar la cola larga del CSV (35.477 filas) **no** eliminó la
> necesidad de escalar: la cola por duración sigue ahí, más corta.

> **Fuga de datos (slide 21 · Error 1):** el escalador se ajusta (`fit`) **solo** sobre la muestra de trabajo; jamás sobre los datos de prueba. Aquí lo hacemos explícito.


In [14]:
# El escalado se hace POR GRUPOS, no a la `X` entera: cada variable recibe el
# escalador que corresponde a lo que es. El detalle de por que esta en el
# docstring de `T.escalar`; aqui queda la evidencia de que salio bien.
esc = T.escalar(df_s)
print("Robust -> mediana despues de escalar (debe ser ~0):",
      esc["numericas"].median().round(4).tolist())
print("MinMax -> min/max despues de escalar (deben ser 0/1):",
      (esc["educacion"].min().round(4).values[0], esc["educacion"].max().round(4).values[0]))
print("Robust (competencias) -> mediana despues de escalar (debe ser ~0):",
      esc["competencias"].median().round(4).tolist())
print("Sin escalar `reincide` porque ya es 0/1; valores:",
      sorted(df_s["reincide"].unique().tolist()))

print("\nEscaladores sobre dur_q (10 primeras de la muestra):")
print(D.comparar_escaladores(df_s).round(3).to_string())
print("\nEl robust mantiene la estructura de la distribucion; el minmax y el")
print("standard quedan a merced de los extremos (cola).")

Robust -> mediana despues de escalar (debe ser ~0): [0.0, 0.0, 0.0]
MinMax -> min/max despues de escalar (deben ser 0/1): (np.float64(0.0), np.float64(1.0))
Robust (competencias) -> mediana despues de escalar (debe ser ~0): [0.0, 0.0]
Sin escalar `reincide` porque ya es 0/1; valores: [0, 1]

Escaladores sobre dur_q (10 primeras de la muestra):
   crudo  minmax  standard  robust
0   16.0   0.077     0.491   1.714
1    1.0   0.000    -0.501  -0.429
2   12.0   0.056     0.226   1.143
3    5.0   0.021    -0.237   0.143
4    1.0   0.000    -0.501  -0.429
5    4.0   0.015    -0.303   0.000
6    1.0   0.000    -0.501  -0.429
7    8.0   0.036    -0.038   0.571
8    1.0   0.000    -0.501  -0.429
9    8.0   0.036    -0.038   0.571

El robust mantiene la estructura de la distribucion; el minmax y el
standard quedan a merced de los extremos (cola).


In [15]:
# POR QUE ESCALAR: cuanto aporta cada columna a la distancia euclidea.
# KNN y K-means eligen vecinos por distancia; como la distancia suma diferencias
# al cuadrado, la columna con mas unidades secuestra la metrica.
pares = D.parejas_aleatorias(len(df_s))
escenarios = D.escenarios_distancia(df_s, esc)
tabla_aporte = D.tabla_aporte(escenarios, pares)

print("Aporte de cada columna a la distancia euclidea (%)")
print(f"Sobre {T.N_PARES} parejas aleatorias de la submuestra de {len(df_s):,} filas\n")
print(tabla_aporte.to_string())

fechas_crudo = tabla_aporte.loc["CRUCO (sin escalar)", ["start_ord", "end_ord"]].sum()
print(f"\nSin escalar, las dos fechas aportan {fechas_crudo:.1f} % de la distancia")
print(f"y la educacion solo {tabla_aporte.loc['CRUCO (sin escalar)', 'univ_ord']:.2f} %:")
print("  -> sin escala, KNN/K-means agruparían por CALENDARIO, no por carrera.")
print(f"\nCon la mezcla de produccion, la duracion del empleo pasa de "
      f"{tabla_aporte.loc['CRUCO (sin escalar)', 'dur_q']:.1f} % a "
      f"{tabla_aporte.loc['PRODUCCION (rob+mm)', 'dur_q']:.1f} % de la distancia.")
print(f"Pero la educacion queda en {tabla_aporte.loc['PRODUCCION (rob+mm)', 'univ_ord']:.2f} %:")
print("  -> efecto lateral de escalar las fechas por IQR (Robust) y la educacion por")
print("     rango total (MinMax). Se documenta como deuda abierta, no se cambia en")
print("     silencio: revisar antes de la fase de mineria (ver seccion 5).")

# El caso concreto que motivo agregar las competencias: sin escalarlas, una
# columna con 94 unidades de rango deja muda a otra que tiene 4.
print("")
print("Aporte de las competencias ESCO a la distancia (seccion 2.5):")
for col in T.SKILLS:
    print(f"  {col:<20} sin escalar {tabla_aporte.loc['CRUCO (sin escalar)', col]:>6.2f} %"
          f"   ->   con Robust {tabla_aporte.loc['PRODUCCION (rob+mm)', col]:>6.2f} %")
print(f"  {'reincide (0/1)':<20} sin escalar "
      f"{tabla_aporte.loc['CRUCO (sin escalar)', 'reincide']:>6.2f} %"
      f"   ->   en produccion {tabla_aporte.loc['PRODUCCION (rob+mm)', 'reincide']:>6.2f} %")
print("  -> el binario no necesita escalador: ya esta en [0, 1]. Aplicarselo lo")
print("     achataria sin ganar nada (regla de dummies y binarias, slide 21).")

# Una pareja concreta: mismo nivel educativo, duraciones lo mas distintas posible.
ia, ib = D.pareja_ilustrativa(df_s)
print(f"\nPareja concreta (mismo nivel educativo, duraciones opuestas):")
print(f"  A: {df_s['start_date'].iloc[ia]} -> {df_s['end_date'].iloc[ia]} | dur_q = {df_s['dur_q'].iloc[ia]:.0f} | "
      f"{df_s['university_level'].iloc[ia]} | {df_s['occupation_label'].iloc[ia]}")
print(f"  B: {df_s['start_date'].iloc[ib]} -> {df_s['end_date'].iloc[ib]} | dur_q = {df_s['dur_q'].iloc[ib]:.0f} | "
      f"{df_s['university_level'].iloc[ib]} | {df_s['occupation_label'].iloc[ib]}")

print("\nDistancia euclidea entre A y B:")
for nombre, Z in escenarios.items():
    print(f"  {nombre:<22}{np.sqrt(((Z[ia] - Z[ib]) ** 2).sum()):>10.3f}")

Aporte de cada columna a la distancia euclidea (%)
Sobre 300 parejas aleatorias de la submuestra de 50,000 filas

                     start_ord  end_ord  dur_q  n_skills_essential  veces_ese_oficio  univ_ord  reincide
CRUCO (sin escalar)      37.85    45.90   7.39                8.76              0.05      0.03      0.01
StandardScaler           15.54    16.71  11.26               16.96             13.32     13.97     12.25
MinMaxScaler              6.84     7.22   1.53                7.82              1.28     15.74     59.57
RobustScaler             12.26    12.95  38.31               13.16             13.31      8.10      1.92
PRODUCCION (rob+mm)      13.26    14.01  41.45               14.24             14.41      0.55      2.07

Sin escalar, las dos fechas aportan 83.8 % de la distancia
y la educacion solo 0.03 %:
  -> sin escala, KNN/K-means agruparían por CALENDARIO, no por carrera.

Con la mezcla de produccion, la duracion del empleo pasa de 7.4 % a 41.5 % de la distancia.
Per

### 4.3 ¿PCA? Auditoría de correlación (slides 12–14)

El pipeline de la sesión 4 cierra con **"PCA, si aplica"**: la clase pide una **correlación** antes de decidir (`pca si o no aplica hay que hacer una correlacion`). PCA busca pocas direcciones (componentes) que expliquen la mayor varianza de columnas **numéricas escaladas**; si las columnas están poco correlacionadas, no hay direcciones que comprimir y PCA **no aplica**.

Aquí lo auditamos con honestidad:

1. **Escalamos** las numéricas con `StandardScaler` (el requisito del slide 12 es PCA sobre datos escalados; caso donde Standard **sí** corresponde).
2. **Correlaciones** de Pearson entre las numéricas.
3. **Varianza acumulada** de las componentes: necesitamos ~80–90 % para justificar comprimir.
4. Las **dummies** (one-hot) quedan fuera: son categóricas puras y ya se "comprimió" convirtiendo categorías en columnas; PCA sobre 2.800 dummies araña información estructural (slide 14).
5. Las **competencias ESCO sí entran** (2 columnas de la 2.5) porque son numéricas continuas. El binario `reincide` queda fuera, por la misma lógica que las dummies: es un indicador 0/1, no una dimensión.


In [16]:
# El PCA entra como DIAGNOSTICO, no como transformacion: se mira si comprime y se
# documenta que no. Las competencias si entran; el binario `reincide` no, por la
# misma razon que las dummies: es un indicador categorico, no una dimension continua.
pca = D.auditoria_pca(df_s, rng)
print("Numericas que entran a PCA:", pca["X_num"].shape[1], "columnas")
print("Correlacion de Pearson (numericas de la muestra):")
print(pca["correlacion"].round(3).to_string())
print()
print("Nota: dur_q = end_ord - start_ord  ->  tres de las columnas guardan una")
print("relacion lineal casi exacta. Las competencias, en cambio, son ortogonales")
print("a las fechas: describen el PUESTO, no el CUANDO.")

print(f"\nVarianza explicada del PCA sobre las {pca['k']} numericas:")
print(pca["varianza"].to_string())
print(f"\nPara llegar a ~80% se necesitan {pca['n_para_80']} de {pca['k']} componentes")

# Que pasa con las dummies y el binario? Solo contamos columnas: no entran a PCA.
print("\nColumnas one-hot de occupation_code:", cod["ocupacion"].shape[1], "-> no entran a PCA")
print("Binario reincide: 1 columna -> tampoco entra (misma logica que las dummies)")
print("Diagnostico: PCA casi no comprime aqui. No se aplica a X final.")

Numericas que entran a PCA: 6 columnas
Correlacion de Pearson (numericas de la muestra):
                    start_ord  end_ord  dur_q  univ_ord  n_skills_essential  veces_ese_oficio
start_ord               1.000    0.875 -0.137     0.084               0.038             0.018
end_ord                 0.875    1.000  0.360     0.074               0.027            -0.006
dur_q                  -0.137    0.360  1.000    -0.009              -0.017            -0.047
univ_ord                0.084    0.074 -0.009     1.000               0.105            -0.026
n_skills_essential      0.038    0.027 -0.017     0.105               1.000             0.094
veces_ese_oficio        0.018   -0.006 -0.047    -0.026               0.094             1.000

Nota: dur_q = end_ord - start_ord  ->  tres de las columnas guardan una
relacion lineal casi exacta. Las competencias, en cambio, son ortogonales
a las fechas: describen el PUESTO, no el CUANDO.

Varianza explicada del PCA sobre las 6 numericas:
      

**Conclusión de la auditoría (misma lógica del resultado de TelecomUNO en el slide 13):** PCA **no comprime** en este dataset. Con 6 numéricas hacen falta **4 de 6 componentes** para llegar al 80 % de la varianza, y la última aporta 0,0000 de varianza individual: el espectro está repartido, no concentrado.

El motivo es que las columnas miden cosas distintas y no se solapan: las fechas y la duración son una **relación lineal casi exacta** entre sí (`dur_q` = `end_ord` − `start_ord`), y las competencias ESCO son **ortogonales** a todo eso — describen el *puesto*, no el *cuándo*. `univ_ord` también aporta su propia dimensión. No hay una dirección dominante que capture la estructura, que es justamente la condición para que PCA sirva.

`PCA` se **descarta** entonces, y por diseño queda excluido de la matriz final (dummies inmensas + columna densa pequeña). Su valor aquí fue **diagnóstico**: confirmó que no hay una estructura de varianza compartida que comprimir sin romper el one-hot.


---
## 5 · Matriz X final lista para modelar

**Formato (espejo de la sesión 4, slide 16):** numéricas/ordinales escaladas + dummies + banderas. De las tres banderas del dataset solo entra `es_vigente`: las otras dos quedan constantes `False` en este universo y se excluyen a propósito (ver 2.5). Nada de texto, nada de binarias escaladas, sin `resume_id`, sin columnas descartadas.


In [17]:
X_final = T.armar_X(esc, cod, df_s, VERSION)
print("X final (submuestra):", X_final.shape)
print("\nPrimeras columnas:", X_final.columns[:12].tolist(), "...")
print("\nComprobaciones de sanidad:")
print(" - dtypes no-numericos:", int(X_final.select_dtypes(exclude=["number"]).shape[1]))
print(" - nulos:", int(X_final.isna().sum().sum()))
X_final.head(3)

X final (submuestra): (50000, 2326)

Primeras columnas: ['start_ord_rob', 'end_ord_rob', 'dur_q_rob', 'n_skills_essential_rob', 'veces_ese_oficio_rob', 'university_mm', 'reincide', 'emparejado_ok', 'occupation_0110.1', 'occupation_0110.10', 'occupation_0110.11', 'occupation_0110.12'] ...

Comprobaciones de sanidad:
 - dtypes no-numericos: 0


 - nulos: 0


,start_ord_rob,end_ord_rob,dur_q_rob,n_skills_essential_rob,veces_ese_oficio_rob,university_mm,reincide,emparejado_ok,occupation_0110.1,occupation_0110.10,...,occupation_9623.1,occupation_9623.2,occupation_9629.1,occupation_9629.2,occupation_9629.3,occupation_9629.4,occupation_9629.5,occupation_9629.6,occupation_9629.7,es_vigente
0,0.142857,0.433333,1.714286,0.307692,0.0,0.25,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.428571,0.200000,-0.428571,0.538462,0.0,0.75,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.428571,-0.233333,1.142857,-0.923077,0.0,0.25,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


> La variable objetivo `y` aún no existe: la minería de **transiciones laborales** (qué ocupaciones siguen a cuáles por persona) se formaliza en la siguiente fase. Por ahora entregamos la **X** que habilitará KNN / K-means / árboles en sesiones 7–9.

### Guardar (opcional)

Descomente la siguiente celda si quiere persistir la matriz. Nota: 50.000×~2.800 columnas densas ocupan ~100–150 MB; para el dataset completo use el bloque `OneHotEncoder` sparse.


In [18]:
# La `X` NO se persiste: son 50.000 x 2.326 (~116 MB en denso) y el curso no pide
# el archivo. Si algun dia hace falta, se descomenta con T.PROC como destino.
# X_final.to_csv(T.PROC / f"X_empleos_{VERSION}_transformado_sample.csv", index=False)

---
## 6 · Resumen del pipeline y cierre

```text
empleos_limpio_sin_outliers.csv (341.090 (sin cola larga) x 20)
  ├─ 1  Ordinal:  fechas        -> anio*4+q -> KBinsDiscretizer (demo) + RobustScaler
  ├─ 2  Ordinal:  university_level -> mapeo manual 0-4 -> MinMaxScaler
  ├─ 3  One-hot:  emparejado (1) + occupation_code (2.855 en el dataset; ~2.300 en la submuestra de 50.000, que es lo que se ve en la salida)  [get_dummies demo / OHE-sparse prod]
  ├─ 4  Binarias: es_vigente (0/1, sin escalar). es_unknown_ocupacion y
  │        es_rescatado quedan constantes False tras el drop y NO entran en la X
  ├─ 5  Derivada: dur_q (duracion trimestral) -> RobustScaler
  ├─ 6  ESCO 2.5: n_skills_essential, veces_ese_oficio -> RobustScaler
  │        reincide (veces > 1) -> 0/1 sin escalar
  │        [se descartan 26.186 filas sin oficio clasificado: 341.090 -> 314.904]
  └─ -  Descartadas (justificado): resume_id, matched_code (fusionado), occupation_label
      (1:1), isco_group / isco_group_label (agregados), isco_level (constante)
```

**Decisiones clave de esta sesión**

1. `isco_level` es ordinal por diseño pero **constante** en los datos → descartada (su `NaN` = `es_unknown_ocupacion`).
2. Redundancias demostradas → `matched_code`, `occupation_label`, `isco_group*` descartados tras auditoría (1:1 / prefijo / fusión).
3. Escaladores elegidos **por los outliers re-auditados en la sección 4.1**: robust para fechas y duración (colas reales), **minmax** para educación (rango acotado sin atípicos) y standard **descartado** por sensibilidad a las colas.
4. Dummies y banderas binarias **no se escalan** (slide 21). `reincide` tampoco: ya es 0/1. De las tres banderas solo entra `es_vigente`: las otras dos son constantes `False` en el universo de trabajo, y una constante no aporta a la distancia ni a la importancia de variables (ver 2.5).
5. **Competencias ESCO (2.5):** de las 6 columnas entran 2 a la `X` más un binario. `n_skills_essential` es función determinista de `occupation_code` (0 de 2.826 oficios con más de un valor), así que su información ya está en el one-hot de ocupación; y `n_skills_competence` / `_knowledge` / `_sin_tipo` son la misma variable repartida, no tres. Se **descartan** de la `X` (siguen en el CSV).
6. **Las 26.186 filas sin oficio clasificado (7,68 %) salen de la `X`**: sus contadores están vacíos —nunca en 0, porque 0 afirmaría que el cargo no exige competencia— y `sklearn` no admite `NaN` en un escalador. El proyecto no imputa, así que se dropean y se declara la pérdida: 341.090 → 314.904 filas, 69.182 → 68.338 personas.
7. `fit` del escalador solo sobre la muestra de trabajo (evita fuga de datos).
8. **PCA auditado y descartado (sección 4.3):** hacen falta 4 de 6 componentes para el 80 % de la varianza y la última no aporta nada, así que no hay una dirección dominante que comprimir. Las fechas y la duración son una dependencia lineal entre sí, y las competencias ESCO son ortogonales a ellas (`n_skills_essential` vs `start_ord` = 0,038). Las dummies y el binario no entran por diseño. No se aplica a `X` final.

**Comparación con la otra versión (4.0 / 4.1):** el pipeline es idéntico; lo que cambia es la magnitud de los outliers y, por tanto, la justificación del robust en `dur_q`.

**Siguiente paso:** sesión 5 (EDA: histogramas, correlaciones, mapa de calor) sobre esta X y las banderas.
